# CoShop Tools & Agent Explorer

This notebook demonstrates the tools available to shopping agents and shows how to run
an end-to-end conversation with the `RawLLM` agent.

**Sections:**
1. Dataset setup
2. Catalog search tool (BM25)
3. Hard-filter catalog search
4. Reflect tool
5. Running a `RawLLM` agent conversation
6. Getting final predictions
7. Inspecting agent state

In [ ]:
%load_ext autoreload
%autoreload 2

import os
# Set your LLM provider API key before running:
# os.environ["ANTHROPIC_API_KEY"] = "..."
# os.environ["OPENAI_API_KEY"] = "..."

## 1. Dataset setup

In [ ]:
from coshop.data import get_dataset

# Choose a dataset: 'movielens', 'goodreads', or 'hm'
# Assets are downloaded from Google Drive on first use for the chosen dataset.
DATASET = 'movielens'
SPEC_IDX = 0

ds = get_dataset(DATASET, max_xstar=1)
spec = ds[SPEC_IDX]

print(f"Dataset: {DATASET}  |  Item type: {spec.item_name}")
print(f"Catalog size: {len(ds.catalog)}")
print(f"xstar: {spec.xstar}")

## 2. Catalog search tool (BM25)

`get_retrieval_fn` returns a callable that wraps BM25 over the catalog representations.
Use `get_retrieval_tool` to get a LangChain tool the agent can call.

In [ ]:
from coshop.tools.retrieval.retrieval import get_retrieval_fn
from coshop.tools.catalog_retrieval import get_retrieval_tool

# Build a BM25 query function over the full catalog
retrieval_fn = get_retrieval_fn(
    "BM25",
    catalog=ds.catalog,
    representation=ds.representation,
)

print("Query function:", retrieval_fn)

In [ ]:
# Call the query function directly (returns a DataFrame)
results = retrieval_fn("action adventure", m=5)
print(f"Returned {len(results)} results")
results

In [ ]:
# With allowed_ids: restrict results to a subset of items
subset_ids = list(ds.catalog.index[:200].astype(str))
results_filtered = retrieval_fn("action adventure", m=5, allowed_ids=subset_ids)
print("Results restricted to first 200 catalog items:")
results_filtered

In [ ]:
# Wrap as a LangChain tool (this is what the agent calls at runtime)
search_tool = get_retrieval_tool(
    "BM25",
    catalog=ds.catalog,
    representation=ds.representation,
    max_items_limit=50,
    execution_max_queries=10,
)

print("Tool name:", search_tool.name)
print("Tool description:\n", search_tool.description)

In [ ]:
# Invoke the tool directly as the agent would
tool_output = search_tool.invoke({"query": "animated family comedy", "max_items": 5})
for item in tool_output[:3]:
    print(item["id"], "|", item["text"][:120], "...")
    print()

## 3. Hard-filter catalog search

`get_hard_filter_retrieval_tool` adds an optional `filters` argument: the agent can pin
exact column values before the semantic search runs.

In [ ]:
from coshop.tools.catalog_retrieval import get_hard_filter_retrieval_tool

# filterable_features are columns the agent is allowed to filter on
filterable_features = ds.filterable_features
print("Filterable features (first 10):", filterable_features[:10])

hard_filter_tool = get_hard_filter_retrieval_tool(
    retrieval_name="BM25",
    catalog=ds.catalog,
    filterable_features=filterable_features,
    representation=ds.representation,
    max_items_limit=50,
    show_col_names=True,   # expose column names + example values in the tool description
)
print("\nTool name:", hard_filter_tool.name)

In [ ]:
# Invoke with a hard filter (e.g., animated=True for movielens)
try:
    filtered_output = hard_filter_tool.invoke({
        "query": "family movie",
        "max_items": 5,
        "filters": {"animated": ["True"]},
    })
    for item in filtered_output[:3]:
        print(item["id"], "|", item["text"][:120], "...")
        print()
except Exception as e:
    print(f"Filter not applicable for this dataset: {e}")

## 4. Reflect tool

The reflect tool lets the agent write out its reasoning before acting.

In [ ]:
from coshop.tools.reflect import get_reflect_tool

reflect = get_reflect_tool()
print("Tool name:", reflect.name)
print("Description:", reflect.description)

# Invoke directly
output = reflect.invoke({"thought": "The user mentioned they like action films. I should search for high-energy titles."})
print("\nReflect output:", output)

## 5. Running a RawLLM agent conversation

`RawLLM` is the baseline conversational agent. It receives a spec, a set of tools,
and a budget, then converses with the user to elicit preferences before making
final predictions.

In [ ]:
from .conversational import RawLLM, MSG_FMT_STRUCTURED_DIALOG_ACTIONS

# MODEL_NAME: any LiteLLM-compatible model string, e.g.:
#   "claude-haiku-4-5-20251001"  (requires ANTHROPIC_API_KEY)
#   "gpt-5-nano"          (requires OPENAI_API_KEY)
MODEL_NAME = "gpt-5-nano"

agent = RawLLM(
    spec=spec,
    model_name=MODEL_NAME,
    msg_fmt_instructions=MSG_FMT_STRUCTURED_DIALOG_ACTIONS,
    actions=[hard_filter_tool],   # tools the agent can use during conversation
    budget_questions=5,
    verbosity=1,   # 0=silent, 1=outputs, 2=full prompts
)
print("Agent initialized.")

In [ ]:
# Start the conversation — agent greets the user
reply = agent()
print("AGENT:", reply)

In [ ]:
# User responds
reply = agent("I'm looking for a fun comedy movie to watch with my kids.")
print("AGENT:", reply)

In [ ]:
# Continue the conversation
reply = agent("Something animated would be great. We've already seen Toy Story.")
print("AGENT:", reply)

In [ ]:
# One more turn
reply = agent("We prefer newer movies, maybe from the 2000s or later.")
print("AGENT:", reply)

## 6. Getting final predictions

After elicitation, call `get_final_predictions` to get the agent's top-k ranked items.
The agent uses a fresh agentic search loop to produce the final list.

In [ ]:
# Get top-5 predictions (agentic mode: agent searches the catalog autonomously)
top_k, metadata = agent.get_final_predictions(
    k=5,
    retrieval_function=retrieval_fn,
    execution_max_per_retrieval=50,
    execution_max_queries=5,
    mode="agentic",
)
print("Top-5 predicted item IDs:", top_k)

# Show titles for movielens
if 'title' in ds.catalog.columns:
    for item_id in top_k:
        if item_id in ds.catalog.index:
            print(f"  {item_id}: {ds.catalog.loc[item_id, 'title']}")

In [ ]:
# Evaluate predictions against ground truth
print("Ground-truth xstar:", spec.xstar)
hits = set(top_k) & set(spec.xstar)
print(f"Hits (xstar in top-k): {hits}")
print(f"Recall@{len(top_k)}: {len(hits) / len(spec.xstar):.2f}")

## 7. Inspecting agent state

In [ ]:
# Full conversation history: LangChain messages in order
for msg in agent.agent_executor.state:
    role = type(msg).__name__.replace('Message', '')
    content = str(msg.content)[:200]
    print(f"[{role}] {content}\n")

In [ ]:
# Elicitation system message the agent received
print(agent.get_elicitation_system_msg())

In [ ]:
# Reset and try a different spec
new_spec = ds[1]
agent.reset(spec=new_spec)
print(f"New spec xstar: {new_spec.xstar}")

reply = agent("Hi, I'm looking for a movie.")
print("AGENT:", reply)